# **Mount Drive**

# **Map Plots 2**

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
formatted_numbers = [int(n) for n in counties['FIPS'].values]
counties['FIPS'] = formatted_numbers
# counties.plot()

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
# df_svi

In [ ]:
fpath1 = '<DATA_ROOT>/WeatherIndex/Temp_Tercile_11nov.csv'
fpath2 = '<DATA_ROOT>/WeatherIndex/Drought_Map.csv'
dft = pd.read_csv(fpath1)
dfd = pd.read_csv(fpath2)
# dfd

dft = dft.rename(columns={'Year':'year'})


dfd = dfd.rename(columns={'Year':'year', 'STATE':'State', 'COUNTY':'County'})
fpath = '<DATA_ROOT>/WeatherIndex/svi2020fips.csv'
df_fips_obj = pd.read_csv(fpath)[['FIPS',	'OBJECTID']]
dfd = pd.merge(dfd,df_fips_obj,on=['OBJECTID'])
dft.iloc[0:5,:]

In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
# cond1 = df_fail['year']>2011
cond2 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond2]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
# df_fail.iloc[0:20,:]

In [ ]:
# ************************************************************************************
# ************************************************************************************
# ************************************************************************************
# ************************************************************************************
# ************************************************************************************

In [ ]:
# counties  df_svi  dfd df_fail
# dfd = dfd.drop(columns=['State', 'County', 'OBJECTID'])
dft = dft.drop(columns=['State', 'County'])
df_svi = df_svi.drop(columns=['State', 'County'])
df_fail = df_fail.drop(columns=['State', 'County', 'Prevented Acres'])

In [ ]:
drought_cols = list(dft.columns)
drought_cols.remove('FIPS')
drought_cols.remove('year')
print(drought_cols)
cond = dft.loc[:,drought_cols] == 0
dft[cond] = 1
dft

In [ ]:
print(drought_cols)


['Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36']


In [ ]:
df_svi = df_svi.set_index(['FIPS','year'])
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
df_svi = df_svi.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi = df_svi.reset_index()
# df_svi

In [ ]:
df_fail_agg = df_fail.groupby(['FIPS','Crop','year','Irrigation Practice']).mean().reset_index()
df_fail_agg
df_merge = pd.merge(df_fail_agg,df_svi,on=['FIPS','year'])
df_merge = pd.merge(df_merge,dft,on=['FIPS','year'])
df_merge

In [ ]:
quantiles_fail = [0, 0.20, 0.40, 0.60, 0.80, 0.99]
cond1 = df_merge['Irrigation Practice'] == 'ALL'
cond2 = df_merge['fail_share'] > 0
df_meany = df_merge[cond1 & cond2].groupby(['Crop','FIPS'])['fail_share'].mean().reset_index()
df_quantiles = df_meany.groupby('Crop')['fail_share'].quantile(q=quantiles_fail).reset_index()
df_quantiles = df_quantiles.pivot(index='level_1', columns='Crop', values='fail_share')
df_quantiles

In [ ]:
df_fail_agg

In [ ]:
df_merge#[['FIPS', 	'Crop', 	'year']]

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import ListedColormap, BoundaryNorm
import gc

matplotlib.use("Agg")

themes = ['RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4','RPL_THEMES']
drought_idxs = ['Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36']
crops = ['CORN', 'COTTON', 'OATS', 'SORGHUM', 'SOYBEANS', 'WHEAT', 'BARLEY', 'HAY']
irig = ['I' , 'N' , 'ALL']
cat_vals = [1,2,3,4]

check_save_fig = False

for theme in themes:
  for dr_idx in drought_idxs:
    df_merge[df_merge[dr_idx]==0] = 1
    for crop in crops:
      cond_crop = df_merge.Crop == crop
      for ig in irig:
        if (crop + '_' + theme + '_' + dr_idx + '_' + ig) == 'OATS_RPL_THEME4_Thr29_I':
          check_save_fig = True
        print(crop + '_' + theme + '_' + dr_idx + '_' + ig , check_save_fig)
        if check_save_fig:
          cond_irig = df_merge['Irrigation Practice'] == ig
          group_cols = ['FIPS', theme, dr_idx]
          df_group = df_merge[cond_crop & cond_irig].groupby(group_cols)['fail_share'].mean().reset_index()
          df_group = df_group[df_group.fail_share>0].dropna(subset=['fail_share'])

          range_values = list(df_quantiles[crop])
          fig, axes = plt.subplots(4, 2, dpi=800)
          i = 0

          for is_drought in [-1,1]:
            if is_drought == -1:
              cond_is_drought = df_group[dr_idx] == -1
              is_drought_label = 'Non-HeatWave'
            else:
              cond_is_drought = df_group[dr_idx] != -1
              is_drought_label = 'HeatWave'

            for cat in cat_vals:
              cond_cat = df_group[theme] == cat
              df_group_i = df_group[(cond_is_drought) & (cond_cat)]
              counties_new = pd.merge(counties,df_group_i[['FIPS','fail_share']],on=['FIPS'],how='left')

              ax = axes[i%4,i//4]
              colors = ['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026']
              cmap = ListedColormap(colors)
              norm = BoundaryNorm(range_values, len(colors))

              counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
              counties_new.plot(ax=ax, column='fail_share', legend=False, cmap=cmap, norm=norm)

              ax.set_xticks([])
              ax.set_yticks([])

              if i == 0 or i == 4:
                ax.set_title(f'{is_drought_label}', fontdict={'fontsize':8})
              ax.set_ylabel('')
              if i < 4:
                ax.set_ylabel(f'{theme}:{cat}', fontdict={'fontsize':6})
              ax.set_xlabel('')
              ax.set_xticklabels([])
              i = i + 1

          cax = fig.add_axes([0.35, 0.05, 0.3, 0.02])
          sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
          sm.set_array([])
          cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
          cbar.ax.tick_params(labelsize=6)
          # cbar.set_label('Your Legend Label')
          plt.subplots_adjust(wspace=-0.45,hspace=0.1)
          plt.suptitle(f'{crop} {theme} Irigation Type:{ig} HeatWave Index:{dr_idx}',fontsize=10)

          out_path = '<DATA_ROOT>/ProcessedData/20231115-CropFailureSVI_CountiesHW-Map/'
          plt.savefig(out_path + crop + '_' + theme + '_' + dr_idx + '_' + ig + '.png' , dpi=1200)
          # plt.close('all')
          plt.clf()
          plt.close(fig)
          del counties_new, df_group
          gc.collect()


# old

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import ListedColormap, BoundaryNorm
%matplotlib inline

themes = ['RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4','RPL_THEMES']
drought_idxs = ['Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36']
crops = ['CORN', 'COTTON', 'OATS', 'SORGHUM', 'SOYBEANS', 'WHEAT', 'BARLEY', 'HAY']
irig = ['I' , 'N' , 'ALL']
cat_vals = [1,2,3,4]

check_save_fig = False

theme = 'RPL_THEME1'
dr_idx = 'Thr26'
df_merge[df_merge[dr_idx]==0] = 1
crop = 'CORN'
cond_crop = df_merge.Crop == crop
ig = 'ALL'

cond_irig = df_merge['Irrigation Practice'] == ig
group_cols = ['FIPS', theme, dr_idx]
df_group = df_merge[cond_crop & cond_irig].groupby(group_cols)['fail_share'].mean().reset_index()
df_group = df_group[df_group.fail_share>0].dropna(subset=['fail_share'])

range_values = list(df_quantiles[crop])
fig, axes = plt.subplots(4, 2, dpi=1200)
i = 0
cond_is_drought = df_group[dr_idx] != -1
is_drought_label = 'Drought Years'
for cat in cat_vals:
  cond_cat = df_group[theme] == cat
  df_group_i = df_group[(cond_is_drought) & (cond_cat)]
  counties_new = pd.merge(counties,df_group_i[['FIPS','fail_share']],on=['FIPS'],how='left')

  ax = axes[i%4,i//4]
  colors = ['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026']
  cmap = ListedColormap(colors)
  norm = BoundaryNorm(range_values, len(colors))

  counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
  # counties_new.plot(ax=ax, column='fail_share', legend=False, cmap=cmap, norm=norm)
  counties_new.plot(ax=ax, column='fail_share', legend=False, cmap="Reds")

  ax.set_xticks([])
  ax.set_yticks([])

  if i == 0 or i == 4:
    ax.set_title(f'{is_drought_label}', fontdict={'fontsize':8})
  ax.set_ylabel('')
  if i < 4:
    ax.set_ylabel(f'{theme}:{cat}', fontdict={'fontsize':6})
  ax.set_xlabel('')
  ax.set_xticklabels([])
  i = i + 1

cax = fig.add_axes([0.35, 0.05, 0.3, 0.02])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.ax.tick_params(labelsize=6)
# cbar.set_label('Your Legend Label')
plt.subplots_adjust(wspace=-0.45,hspace=0.1)
plt.suptitle(f'{crop} {theme} Irigation Type:{ig} Drought Index:{dr_idx}',fontsize=10)

plt.show()
plt.close()




Output hidden; open in https://colab.research.google.com to view.